# v1 commissioning (read-only)

One section per implementation stage (S0–S9). Cells only call functions from `twobunch_deploy`; each stage saves its records under `data/` (git-ignored, copied back by hand), and the result goes in the lab-notebook commissioning page.

Nothing here writes to the machine: `tests/test_read_only.py` fails on any write call in the package or in these cells.

## S0: environment

Reads no PV. Run on the host in the ACR prod Python environment, from the repo root (Jupyter started inside tmux).

Pass:
- `import_errors` is empty;
- `code_sha256` equals the value CI printed for this PR's head commit (the `pixi run code-hash` step);
- `git_commit` is that commit;
- `epics_env` shows the host's Channel Access settings (recorded; no target value).

In [ ]:
from twobunch_deploy import session

env = session.check_env()
env

In [ ]:
session.save_record("S0", "check_env", env)

## S1: camera frames and pulse IDs

Reads one PV, `CAMR:LI20:107:Image:ArrayData` (the DTOTR2 camera image): subscribes for 30 s, then disconnects. Needs the camera on, and beam for real images. Saves the summary as JSON and, next to it (`.npz`), every frame's pulse ID and timestamps plus 10 raw frames.

Pass:
- `frames` > 0 and `latency_s.first` under about 1 s: the camera is publishing, and the frame timestamps agree with this host's clock;
- `lengths` is a single value: 164,496 (184 × 894) if the ROI is still TEST_15668's (S2 checks the geometry);
- `min` ≥ 0 and `max` ≤ 4095 (12-bit camera); `dtype` recorded;
- `pulse_id_steps` almost all 36. A 0 next to a 72, or a 35 next to a 37, is the IOC stamping a frame with a neighbouring pulse ID (8 times in 33,779 DAQ frames); any other step is a finding;
- `capture_fraction` close to 1: the callback should get every frame. Lower means frames were lost between the IOC and the notebook.

In [ ]:
from twobunch_deploy import session

summary = session.capture_test(seconds=30)
summary